## QTM350 Final Project Data Cleaning

## Research Question: <br> Do higher education levels and GDP per capita contribute to lower adolescent fertility rates across countries?

#### First we create a python conda environment to use for the rest of this project for reproducibility

In [2]:
!conda create -y -n wb-analysis-env -c conda-forge python=3.10 ipykernel=6.29.3 numpy=1.26.4 pandas=2.2.1 matplotlib=3.8.4 seaborn=0.13.2 wbgapi=1.0.12 requests=2.31.0 sqlalchemy=2.0.29
!python -m ipykernel install --user --name wb-analysis-env --display-name "Python (wb-analysis-env)"
!conda activate wb-analysis-env

Retrieving notices: done
Channels:
 - conda-forge
 - defaults
Platform: osx-arm64
Solving environment: done

## Package Plan ##

  environment location: /opt/anaconda3/envs/wb-analysis-env

  added / updated specs:
    - ipykernel=6.29.3
    - matplotlib=3.8.4
    - numpy=1.26.4
    - pandas=2.2.1
    - python=3.10
    - requests=2.31.0
    - seaborn=0.13.2
    - sqlalchemy=2.0.29
    - wbgapi=1.0.12


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    libpng-1.6.58              |       h132b30e_0         283 KB  conda-forge
    packaging-26.1             |     pyhc364b38_0          87 KB  conda-forge
    ------------------------------------------------------------
                                           Total:         370 KB

The following NEW packages will be INSTALLED:

  _openmp_mutex      conda-forge/osx-arm64::_openmp_mutex-4.5-7_kmp_llvm 
  appnope            conda-forge/noarch::a

#### We import the wbgapi package to fetch data directly from the World Bank and standard libraries for file handling.

#### Data fetch - for 5-year-duration

In [2]:
import wbgapi as wb
import pandas as pd
import sqlite3

# 1. Define the indicators
indicators = {
    'SP.ADO.TFRT': 'adolescent_fertility',
    'SE.PRM.ENRR': 'primary_education',
    'SE.SEC.ENRR': 'secondary_education',
    'SE.TER.ENRR': 'tertiary_education',
    'NY.GDP.PCAP.CD': 'gdp_per_capita'
}

# 2. Fetch data specifically for 2020-2024
# Using columns='series' ensures the indicators become the columns
# labels=True gives us the readable country names
df = wb.data.DataFrame(indicators.keys(), time=range(2020, 2025), labels=True, columns='series').reset_index()

# 3. Clean up the column names
# The columns will likely be named 'economy', 'time', and the indicator codes
df = df.rename(columns={
    'economy': 'country_code',
    'CountryName': 'Country',
    'time': 'year',
    'NY.GDP.PCAP.CD': 'gdp_per_capita',
    'SE.PRM.ENRR': 'primary_education',
    'SE.SEC.ENRR': 'secondary_education',
    'SE.TER.ENRR': 'tertiary_education',
    'SP.ADO.TFRT': 'adolescent_fertility'
})

# 4. Save to SQL
conn = sqlite3.connect(':memory:')
df.to_sql('wdi_raw', conn, index=False, if_exists='replace')

# 5. Look at the result 
df.head(10)

,country_code,year,Country,Time,gdp_per_capita,primary_education,secondary_education,tertiary_education,adolescent_fertility
0,ZWE,YR2024,Zimbabwe,2024,2497.203322,93.077211,NaN,7.74913,95.524
1,ZWE,YR2023,Zimbabwe,2023,2195.224921,93.238379,NaN,NaN,98.057
2,ZWE,YR2022,Zimbabwe,2022,2536.400502,93.165966,NaN,NaN,99.484
3,ZWE,YR2021,Zimbabwe,2021,2613.605421,92.784691,NaN,NaN,99.199
4,ZWE,YR2020,Zimbabwe,2020,2059.674454,93.612586,NaN,NaN,98.893
5,ZMB,YR2024,Zambia,2024,1187.109434,115.550919,56.788929,NaN,115.479
6,ZMB,YR2023,Zambia,2023,1330.727806,NaN,NaN,NaN,115.907
7,ZMB,YR2022,Zambia,2022,1447.123101,NaN,NaN,NaN,117.398
8,ZMB,YR2021,Zambia,2021,1127.160779,NaN,NaN,NaN,118.744
9,ZMB,YR2020,Zambia,2020,951.644317,NaN,NaN,NaN,120.712


In [3]:
query = """
SELECT 
    Country,
    AVG(gdp_per_capita) AS avg_gdp,
    AVG(primary_education) AS avg_pri_edu,
    AVG(secondary_education) AS avg_sec_edu,
    AVG(tertiary_education) AS avg_tert_edu,
    AVG(adolescent_fertility) AS avg_fertility
FROM wdi_raw
WHERE Country NOT LIKE '%income%' 
GROUP BY Country
HAVING avg_gdp IS NOT NULL;
"""
pd.read_sql(query, conn).head(10)

,Country,avg_gdp,avg_pri_edu,avg_sec_edu,avg_tert_edu,avg_fertility
0,Afghanistan,409.575581,112.230904,59.613602,10.854360,65.276800
1,Africa Eastern and Southern,1556.036202,102.126631,45.175237,8.686865,93.455051
2,Africa Western and Central,1907.064196,89.538686,45.736360,10.062820,95.914560
3,Albania,8427.195098,99.745033,99.081844,66.616413,13.004200
4,Algeria,4797.574513,106.245699,103.659765,54.189083,9.100200
5,American Samoa,15914.292694,NaN,NaN,NaN,34.279400
6,Andorra,43663.387382,93.386685,99.026561,39.080248,3.405200
7,Angola,2665.477712,87.981001,52.309529,10.002132,141.983600
8,Antigua and Barbuda,19572.081585,116.909307,108.340344,NaN,33.229200
9,Arab World,7096.978884,89.974500,68.457329,33.298216,45.522724


In [4]:
conn.execute("CREATE VIEW IF NOT EXISTS cleaned_country_data AS " + query)

#### Data Cleaning

In [8]:
# Create a "Long" version for time-series analysis
long_query = """
SELECT 
    Country,
    Year,
    gdp_per_capita AS gdp,
    primary_education AS primary_edu,
    secondary_education AS secondary_edu,
    adolescent_fertility AS fertility
FROM wdi_raw
WHERE Country IN ('Brazil', 'Switzerland', 'Italy', 'Malaysia', 'Niger', 'India')
ORDER BY Country, Year;
"""

df_long = pd.read_sql(long_query, conn)

print("df_long created! Dimensions:", df_long.shape)
display(df_long.head(10))

df_long created! Dimensions: (30, 6)


,Country,year,gdp,primary_edu,secondary_edu,fertility
0,Brazil,YR2020,7074.194075,105.912231,103.882660,47.460
1,Brazil,YR2021,7972.536961,103.929398,106.298462,45.670
2,Brazil,YR2022,9281.332821,104.667419,105.702370,42.881
3,Brazil,YR2023,10377.589772,NaN,NaN,42.686
4,Brazil,YR2024,10310.548878,104.296913,104.144661,41.459
5,India,YR2020,1907.042516,101.291254,77.627155,14.423
6,India,YR2021,2239.613844,102.316124,80.624640,14.289
7,India,YR2022,2347.448294,111.084456,81.177861,14.185
8,India,YR2023,2530.120313,112.031232,78.863959,14.060
9,India,YR2024,2694.737809,120.520369,78.105807,13.036


#### Data Cleaning - creating the view

In [13]:
# Create the cleaned view inside SQL
conn.execute("DROP VIEW IF EXISTS cleaned_research_data")
conn.execute("""
CREATE VIEW cleaned_research_data AS
SELECT 
    Country,
    AVG(gdp_per_capita) AS gdp,
    AVG(primary_education) AS primary_edu,
    AVG(secondary_education) AS secondary_edu,
    AVG(tertiary_education) AS tertiary_edu,
    AVG(adolescent_fertility) AS fertility
FROM wdi_raw
WHERE Country IN ('Brazil', 'Switzerland', 'Italy', 'Malaysia', 'Niger', 'India')
GROUP BY Country
HAVING gdp IS NOT NULL 
   AND fertility IS NOT NULL;
""")
print("Cleaned view created successfully.")

Cleaned view created successfully.


#### Save the data

In [14]:
import os

# 1. Define your directory (ensure it exists)
DATA_DIR = '/Users/sissili/Desktop/QTM350/QTM_350_Final_Project/QTM350_Final_Project_Data' 
if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)

# 2. Pull the final cleaned data from your SQL View into a DataFrame
df_clean = pd.read_sql("SELECT * FROM cleaned_research_data", conn)

# 3. Define the output path and save
output_path = os.path.join(DATA_DIR, 'cleaned_final_project_data.csv')
df_clean.to_csv(output_path, index=False)

print(f"Data successfully exported to: {output_path}")

output_path_long = os.path.join(DATA_DIR, 'cleaned_final_project_data_long.csv')
df_long.to_csv(output_path_long, index=False)
print(f"Long data exported to: {output_path_long}")

Data successfully exported to: /Users/sissili/Desktop/QTM350/QTM_350_Final_Project/QTM350_Final_Project_Data/cleaned_final_project_data.csv
Long data exported to: /Users/sissili/Desktop/QTM350/QTM_350_Final_Project/QTM350_Final_Project_Data/cleaned_final_project_data_long.csv
